Below is the version changing "closest to the middle" rule to "maximize omega" rule. The modified code of heuristic will Identify Widest Gaps: It first determines the values of the N-1 widest gaps. Create Candidate Pool: It then collects all indices of score gaps that have values equal to or greater than the smallest value among these N-1 widest gaps. This creates a pool of "candidate" cut points, ensuring all ties for the widest gaps are included. Maximize Omega from Candidates: From this refined pool of candidate cut indices, it iterates through all combinations of N-1 cuts. For each combination, it calculates the resulting Omega value and selects the combination that yields the highest Omega.

(As of 5 Apr 2026) Below is the copy of the right above to generate the separate output files of all three algorithms to go through manual conditional discretization process.

In [4]:
#(As of 5 Apr 2026)
#Adjust grade symbols at code lines marked with ***
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.stats import zscore
import io # Import io for handling file-like objects
from itertools import combinations # Import combinations for generating cut point combinations

base_output = './file/output'

import os
os.makedirs(f'{base_output}/kmean', exist_ok=True)
os.makedirs(f'{base_output}/zscore', exist_ok=True)
os.makedirs(f'{base_output}/heuristic', exist_ok=True)


# Only import these if running in Colab environment
try:
    from google.colab import files
except ImportError:
    files = None


def calculate_omega(scores_grades_df, grade_symbols, all_sorted_scores):
    """
    Calculates the Omega (Ω) fairness metric as defined in Equation (1) of the paper.

    Args:
        scores_grades_df (pd.DataFrame): DataFrame with 'Score' and 'Grade' columns.
        grade_symbols (list): A list of grade symbols in ranked order (e.g., ['A', 'B', 'C', 'D', 'F']).
        all_sorted_scores (np.array): All original scores, sorted in descending order.

    Returns:
        float: The calculated Omega value, or 0.0 if calculation is not possible (e.g., division by zero).
    """
    N = len(grade_symbols) # Number of eligible unique grade symbols

    # Debug print: Show scores and grades for Omega calculation
    # print("\n--- Omega Calculation Details ---")
    # print("Scores and Grades for Omega calculation:")
    # print(scores_grades_df)

    # 1. Calculate score intervals for each grade and their standard deviation (sigma)
    grade_intervals = []
    for grade in grade_symbols:
        grade_scores = scores_grades_df[scores_grades_df['Grade'] == grade]['Score']
        if grade_scores.size > 0: # Corrected check for numpy array emptiness
            interval = grade_scores.max() - grade_scores.min()
            grade_intervals.append(interval)
        # else:
            # print(f"Warning: No scores found for grade '{grade}' during interval calculation.")

    if len(grade_intervals) < N:
        # print(f"Warning: Not all {N} grade symbols were assigned. Sigma calculation might be affected.")
        pass # Suppress warning for internal heuristic calls

    sigma = np.std(grade_intervals) if grade_intervals else 0.0
    # print(f"Grade Intervals: {grade_intervals}")
    # print(f"Sigma (Std Dev of Intervals): {sigma:.3f}")

    # 2. Calculate delta_i (score gap between adjacent grade symbols)
    # delta_i: score gap between lower bound of grade_i and upper bound of grade_{i+1}
    sum_delta_i = 0.0
    deltas_i_list = []
    for i in range(N - 1):
        better_grade = grade_symbols[i]
        worse_grade = grade_symbols[i+1]

        lower_bound_better_grade_scores = scores_grades_df[scores_grades_df['Grade'] == better_grade]['Score']
        upper_bound_worse_grade_scores = scores_grades_df[scores_grades_df['Grade'] == worse_grade]['Score']

        if lower_bound_better_grade_scores.size > 0 and upper_bound_worse_grade_scores.size > 0: # Corrected check
            lower_bound_better = lower_bound_better_grade_scores.min()
            upper_bound_worse = upper_bound_worse_grade_scores.max()
            delta = lower_bound_better - upper_bound_worse
            deltas_i_list.append(delta)
            sum_delta_i += max(0, delta) # Gaps should be non-negative
        # else:
            # print(f"Warning: Missing scores for grade '{better_grade}' or '{worse_grade}' for delta_i calculation.")
            # deltas_i_list.append(0) # Or handle as per paper's unstated rule for missing grades

    # print(f"Delta_i values (lower_bound_better - upper_bound_worse): {deltas_i_list}")
    # print(f"Sum of Delta_i (non-negative): {sum_delta_i:.3f}")

    # 3. Calculate all score gaps from the original sorted scores
    # These are used for Delta_min and Delta_max
    score_gaps = np.diff(all_sorted_scores)
    score_gaps = np.abs(score_gaps) # Ensure gaps are positive
    # print(f"All original score gaps: {score_gaps}")

    if len(score_gaps) < N - 1:
        # print(f"Warning: Not enough score gaps ({len(score_gaps)}) for {N-1} required for Omega calculation.")
        return 0.0

    # Sort gaps to find narrowest and widest
    sorted_gaps = np.sort(score_gaps) # Ascending order

    # Delta_i_min and Delta_i_max are the i-th narrowest/widest *overall* gaps
    # The paper implies summing N-1 of these.
    # We need to ensure we have enough unique gaps or handle duplicates appropriately.
    # For simplicity, we'll take the N-1 smallest/largest values from the sorted gaps.

    # Sum of N-1 narrowest gaps
    sum_delta_min = np.sum(sorted_gaps[:N-1])

    # Sum of N-1 widest gaps (reverse sort and take first N-1)
    sum_delta_max = np.sum(sorted_gaps[::-1][:N-1])

    # print(f"Sorted all score gaps: {sorted_gaps}")
    # print(f"Sum of N-1 narrowest gaps (Delta_min): {sum_delta_min:.3f}")
    # print(f"Sum of N-1 widest gaps (Delta_max): {sum_delta_max:.3f}")

    # Ensure denominator is not zero
    denominator = (1 + sigma) * (sum_delta_max - sum_delta_min)
    # print(f"Denominator: (1 + {sigma:.3f}) * ({sum_delta_max:.3f} - {sum_delta_min:.3f}) = {denominator:.3f}")

    if denominator == 0:
        # print("Error: Denominator for Omega calculation is zero. Returning 0.0.")
        return 0.0 # Avoid division by zero

    omega = (sum_delta_i - sum_delta_min) / denominator
    # print(f"Calculated Omega: ({sum_delta_i:.3f} - {sum_delta_min:.3f}) / {denominator:.3f} = {omega:.3f}")
    # print("--- End Omega Calculation Details ---\n")
    return omega

def assign_grades(scores, grade_boundaries, grade_symbols):
    """
    Assigns grades based on score boundaries.

    Args:
        scores (pd.Series): Series of scores.
        grade_boundaries (list): List of tuples (min_score, max_score, grade_symbol).
                                 Assumes boundaries are sorted from highest grade to lowest.
        grade_symbols (list): A list of grade symbols in ranked order (e.g., ['A', 'B', 'C', 'D', 'F']).

    Returns:
        pd.DataFrame: DataFrame with 'Score' and 'Grade' columns.
    """
    graded_data = []

    # Sort grade_boundaries based on the upper score bound in descending order
    # This ensures that higher grades (with higher score ranges) are checked first.
    grade_boundaries.sort(key=lambda x: x[1], reverse=True)

    for score in scores:
        assigned_grade = 'F' # Default to lowest grade if no match
        for min_s, max_s, grade_s in grade_boundaries:
            # Check if score falls within the grade's range (inclusive)
            if min_s <= score <= max_s:
                assigned_grade = grade_s
                break # Assign the first matching grade (highest grade if ranges overlap)
        graded_data.append({'Score': score, 'Grade': assigned_grade})
    return pd.DataFrame(graded_data)

def grading_by_kmeans(scores, grade_symbols):
    """
    Grades scores using the K-means clustering method.

    Args:
        scores (np.array): Sorted scores.
        grade_symbols (list): A list of grade symbols in ranked order.

    Returns:
        pd.DataFrame: DataFrame with 'Score' and 'Grade' columns.
    """
    n_clusters = len(grade_symbols)
    # Reshape scores for KMeans (expects 2D array)
    scores_reshaped = scores.reshape(-1, 1)

    # Initialize KMeans with n_init='auto' for modern scikit-learn
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    kmeans.fit(scores_reshaped)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_.flatten()

    # Map clusters to grades based on centroid values (higher centroid = better grade)
    sorted_cluster_indices = np.argsort(centroids)[::-1] # Indices of clusters sorted by centroid (descending)

    grade_map = {sorted_cluster_indices[i]: grade_symbols[i] for i in range(n_clusters)}

    grades = [grade_map[label] for label in labels]

    # Create DataFrame to calculate boundaries for Omega
    temp_df = pd.DataFrame({'Score': scores, 'Grade': grades})

    # Determine grade boundaries for assign_grades (not strictly needed here as we have labels)
    # But for consistency with other methods, let's derive them.
    grade_boundaries = []
    for i, grade_s in enumerate(grade_symbols):
        grade_scores = temp_df[temp_df['Grade'] == grade_s]['Score']
        if grade_scores.size > 0: # Corrected check for numpy array emptiness
            min_s = grade_scores.min()
            max_s = grade_scores.max()
            grade_boundaries.append((min_s, max_s, grade_s))

    # Debug print:
    print(f"KMeans Grade Boundaries (derived from assigned scores): {grade_boundaries}")

    final_df = assign_grades(scores, grade_boundaries, grade_symbols)
    return final_df


def grading_by_zscore(scores, grade_symbols):
    """
    Grades scores using the Z-score method.

    Args:
        scores (np.array): Sorted scores.
        grade_symbols (list): A list of grade symbols in ranked order.

    Returns:
        pd.DataFrame: DataFrame with 'Score' and 'Grade' columns.
    """
    # Calculate Z-scores
    z_scores = zscore(scores)

    # Convert Z-scores to T-scores (mean 50, std dev 10)
    t_scores = 10 * z_scores + 50

    # Determine the range of T-scores
    min_t = np.min(t_scores)
    max_t = np.max(t_scores)
    t_range = max_t - min_t

    n_grades = len(grade_symbols)

    # Divide the T-score range into N equal intervals
    interval_size = t_range / n_grades

    # Define T-score boundaries for each grade
    t_score_boundaries = []
    for i in range(n_grades):
        # Grades are assigned from highest (A) to lowest (F)
        # So, the first interval is for 'A', second for 'B', etc.
        # The T-score range is typically higher for better grades.

        # Calculate upper bound for the current grade's T-score range
        # Start from max_t and go downwards
        upper_bound_t = max_t - (i * interval_size)
        lower_bound_t = max_t - ((i + 1) * interval_size)

        # Adjust for floating point precision for the lowest boundary
        if i == n_grades - 1:
            lower_bound_t = min_t # Ensure the lowest grade covers the absolute minimum T-score

        t_score_boundaries.append((lower_bound_t, upper_bound_t, grade_symbols[i]))

    # Sort boundaries by lower bound of T-score in ascending order to match assign_grades logic
    # (assign_grades expects boundaries from lowest score range to highest, or highest to lowest if iterating in order)
    # Let's ensure assign_grades iterates from highest grade to lowest.
    t_score_boundaries.sort(key=lambda x: x[1], reverse=True) # Sort by upper bound descending

    # Assign grades based on T-score boundaries
    grades = []
    for i, score in enumerate(scores):
        assigned_grade = 'F' # Default to lowest grade
        for lower_t, upper_t, grade_s in t_score_boundaries:
            if lower_t <= t_scores[i] <= upper_t:
                assigned_grade = grade_s
                break
        grades.append(assigned_grade)

    # Create DataFrame to calculate boundaries for Omega
    temp_df = pd.DataFrame({'Score': scores, 'Grade': grades})

    # Determine grade boundaries in terms of original scores for assign_grades
    grade_boundaries_original_scores = []
    for i, grade_s in enumerate(grade_symbols):
        grade_scores = temp_df[temp_df['Grade'] == grade_s]['Score']
        if grade_scores.size > 0: # Corrected check for numpy array emptiness
            min_s = grade_scores.min()
            max_s = grade_scores.max()
            grade_boundaries_original_scores.append((min_s, max_s, grade_s))

    # Debug print:
    print(f"Zscore Grade Boundaries (derived from assigned scores): {grade_boundaries_original_scores}")

    final_df = assign_grades(scores, grade_boundaries_original_scores, grade_symbols)
    return final_df


def grading_by_heuristic(scores, grade_symbols):
    """
    Grades scores using the heuristic method.
    If there are many equal widest score gaps, it chooses the gaps that maximize Omega.

    Args:
        scores (np.array): Sorted scores (descending).
        grade_symbols (list): A list of grade symbols in ranked order.

    Returns:
        pd.DataFrame: DataFrame with 'Score' and 'Grade' columns that yields the highest Omega.
    """
    n_grades = len(grade_symbols)

    # Calculate all possible cut indices (between scores)
    # There are len(scores) - 1 possible places to make a cut.
    gaps = np.abs(np.diff(scores))

    if len(gaps) == 0:
        return pd.DataFrame({'Score': scores, 'Grade': [grade_symbols[0]] * len(scores)})

    num_cuts_needed = n_grades - 1
    if num_cuts_needed <= 0: # Handle cases with 1 or no grades
        return pd.DataFrame({'Score': scores, 'Grade': [grade_symbols[0]] * len(scores)})

    if num_cuts_needed > len(gaps):
        print("Warning: Not enough scores to create all required grade divisions for Heuristic method.")
        return pd.DataFrame({'Score': scores, 'Grade': [grade_symbols[0]] * len(scores)})

    # Create a list of (gap_value, original_index_of_first_score_in_gap)
    indexed_gaps = [(gaps[i], i) for i in range(len(gaps))]

    # Sort gaps by value in descending order
    indexed_gaps.sort(key=lambda x: x[0], reverse=True)

    # Determine the value of the (num_cuts_needed)-th widest gap
    # This handles cases where there are fewer unique gaps than num_cuts_needed
    # or when there are many ties for the widest gaps.
    if len(indexed_gaps) >= num_cuts_needed:
        threshold_gap_value = indexed_gaps[num_cuts_needed - 1][0]
    else:
        threshold_gap_value = 0 # If not enough gaps, consider any gap above 0

    # Create a pool of candidate cut indices: all indices whose gap value is >= threshold_gap_value
    candidate_cut_indices_pool = []
    for gap_val, idx in indexed_gaps:
        if gap_val >= threshold_gap_value:
            candidate_cut_indices_pool.append(idx)
        else:
            # Since indexed_gaps is sorted by gap_value descending,
            # we can break early if we've passed the threshold.
            break

    # Ensure unique indices and sort them
    candidate_cut_indices_pool = sorted(list(set(candidate_cut_indices_pool)))

    # If the pool of candidate indices is still too small to form `num_cuts_needed` combinations,
    # it means there aren't enough distinct "widest" gaps. In such cases, we expand the pool
    # to include more gaps until we have at least `num_cuts_needed` unique indices to choose from.
    # This ensures `combinations` can always find enough sets of cuts.
    if len(candidate_cut_indices_pool) < num_cuts_needed:
        all_possible_cut_indices = list(range(len(scores) - 1))
        # If the number of available gaps is less than cuts needed, we must use all available.
        if len(all_possible_cut_indices) < num_cuts_needed:
             print("Error: Not enough unique score gaps to form required grade divisions.")
             return pd.DataFrame({'Score': scores, 'Grade': [grade_symbols[-1]] * len(scores)})
        candidate_cut_indices_pool = all_possible_cut_indices
        print("Warning: Expanded candidate cut pool as initial 'widest' filter was too restrictive.")


    best_omega = -1.0
    best_grades_df = None

    # Iterate through all combinations of num_cuts_needed from the candidate pool
    # This is the core of "maximize Omega" among the "widest" candidates.
    for cut_combination in combinations(candidate_cut_indices_pool, num_cuts_needed):
        current_cut_indices = sorted(list(cut_combination))

        temp_grade_boundaries = []
        current_start_idx = 0
        valid_combination = True

        for i in range(n_grades):
            grade_s = grade_symbols[i]

            if i < num_cuts_needed:
                end_idx = current_cut_indices[i]
            else:
                end_idx = len(scores) - 1

            current_grade_scores = scores[current_start_idx : end_idx + 1]

            if current_grade_scores.size > 0:
                min_s = current_grade_scores.min()
                max_s = current_grade_scores.max()
                temp_grade_boundaries.append((min_s, max_s, grade_s))
            else:
                # If a grade segment is empty with these cuts, this combination is invalid
                # and will likely lead to a low or zero Omega.
                valid_combination = False
                break

            current_start_idx = end_idx + 1

        if not valid_combination:
            continue

        current_grades_df = assign_grades(scores, temp_grade_boundaries, grade_symbols)
        current_omega = calculate_omega(current_grades_df, grade_symbols, scores)

        # If current_omega is greater than best_omega, update.
        # If current_omega is equal to best_omega, the first one encountered (due to combinations order) is kept.
        if current_omega > best_omega:
            best_omega = current_omega
            best_grades_df = current_grades_df.copy() # Store a copy

    if best_grades_df is None:
        print("Warning: No valid heuristic grading found that yields non-empty grades. Returning default 'F' grades.")
        return pd.DataFrame({'Score': scores, 'Grade': [grade_symbols[-1]] * len(scores)})

    print(f"Heuristic Best Omega Found (from widest gaps): {best_omega:.3f}")
    # print(f"Heuristic Best Grade Boundaries (derived from assigned scores):")
    # print(best_grades_df.groupby('Grade')['Score'].agg(['min', 'max']))

    return best_grades_df


def hybrid_grading_algorithm(input_df, grade_symbols=['A', 'B+', 'B', 'C+', 'C', 'D+', 'D']): #***
    """
    Implements the hybrid grading algorithm, generating separate Excel files for each method.

    Args:
        input_df (pd.DataFrame): DataFrame containing scores.
        grade_symbols (list): A list of grade symbols in ranked order (e.g., ['A', 'B', 'C', 'D', 'F']).

    Returns:
        dict: A dictionary containing file paths of generated Excel files and Omega scores.
    """
    output_files = {}
    omega_results = {}

    try:
        score_column_name = input_df.columns[0]
        scores = input_df[score_column_name].values
        sorted_scores = np.sort(scores)[::-1]

        # 1. K-means Grading
        print("Running K-means grading...")
        kmeans_grades_df = grading_by_kmeans(sorted_scores, grade_symbols)
        omega_kmeans = calculate_omega(kmeans_grades_df, grade_symbols, sorted_scores)
        omega_results['K-means'] = omega_kmeans
        print(f"K-means Omega: {omega_kmeans:.3f}")

        kmeans_output_file = f'{base_output}/kmean/graded_scores_kmeans.xlsx'
        kmeans_grades_df_final = kmeans_grades_df.copy()
        kmeans_grades_df_final = pd.concat([kmeans_grades_df_final,
                                            pd.DataFrame({score_column_name: [f"K-means Omega: {omega_kmeans:.3f}"], 'Grade': ['']})],
                                           ignore_index=True)
        kmeans_grades_df_final.to_excel(kmeans_output_file, index=False)
        output_files['K-means'] = kmeans_output_file
        print(f"K-means grading output saved to '{kmeans_output_file}'")

        # 2. Z-score Grading
        print("Running Z-score grading...")
        zscore_grades_df = grading_by_zscore(sorted_scores, grade_symbols)
        omega_zscore = calculate_omega(zscore_grades_df, grade_symbols, sorted_scores)
        omega_results['Z-score'] = omega_zscore
        print(f"Z-score Omega: {omega_zscore:.3f}")

        zscore_output_file = f'{base_output}/zscore/graded_scores_zscore.xlsx'
        zscore_grades_df_final = zscore_grades_df.copy()
        zscore_grades_df_final = pd.concat([zscore_grades_df_final,
                                            pd.DataFrame({score_column_name: [f"Z-score Omega: {omega_zscore:.3f}"], 'Grade': ['']})],
                                           ignore_index=True)
        zscore_grades_df_final.to_excel(zscore_output_file, index=False)
        output_files['Z-score'] = zscore_output_file
        print(f"Z-score grading output saved to '{zscore_output_file}'")

        # 3. Heuristic Grading (now maximizes Omega among widest gaps)
        print("Running Heuristic grading (maximizing Omega among widest gaps)... This may take a moment for large datasets.")
        heuristic_grades_df = grading_by_heuristic(sorted_scores, grade_symbols)
        omega_heuristic = calculate_omega(heuristic_grades_df, grade_symbols, sorted_scores)
        omega_results['Heuristic'] = omega_heuristic
        print(f"Heuristic Omega: {omega_heuristic:.3f}")

        heuristic_output_file = f'{base_output}/heuristic/graded_scores_heuristic.xlsx'
        heuristic_grades_df_final = heuristic_grades_df.copy()
        heuristic_grades_df_final = pd.concat([heuristic_grades_df_final,
                                               pd.DataFrame({score_column_name: [f"Heuristic Omega: {omega_heuristic:.3f}"], 'Grade': ['']})],
                                              ignore_index=True)
        heuristic_grades_df_final.to_excel(heuristic_output_file, index=False)
        output_files['Heuristic'] = heuristic_output_file
        print(f"Heuristic grading output saved to '{heuristic_output_file}'")

        # Select the method with the highest Omega value
        best_method = None
        max_omega = -1.0
        for method_name, omega in omega_results.items():
            if omega > max_omega:
                max_omega = omega
                best_method = method_name

        print(f"\nSelected best method: {best_method} with Omega: {max_omega:.3f}")

        return {
            'output_files': output_files,
            'omega_results': omega_results,
            'best_method': best_method,
            'best_omega': max_omega
        }

    except KeyError:
        print(f"Error: Could not find a suitable score column in the input DataFrame. "
              "Ensure the first column contains scores and has a header.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# Example Usage for Colab:
if __name__ == "__main__":
    # This part handles file upload in Colab
    if files is not None:
        print("Please upload your Excel file:")
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded. Exiting.")
        else:
            # Assuming only one file is uploaded for simplicity
            input_file_name = list(uploaded.keys())[0]
            print(f'User uploaded file "{input_file_name}" with length {len(uploaded[input_file_name])} bytes')

            # Read the uploaded file into a pandas DataFrame
            input_df_from_upload = pd.read_excel(io.BytesIO(uploaded[input_file_name]))

            # Call the hybrid grading algorithm and get output file paths; #Adjust the grading symbols down here !!!
            grading_results = hybrid_grading_algorithm(input_df_from_upload, grade_symbols=['A', 'B+', 'B', 'C+', 'C', 'D+', 'D']) #***

            if grading_results:
                print("\nDownloading generated files:")
                for method, file_path in grading_results['output_files'].items():
                    print(f"- {method} results: {file_path}")
                    files.download(file_path)
            else:
                print("Grading failed, no files to download.")

    else:
        print("Running in local environment...")

        # 📥 input file (ของคุณอยู่ใน file/input/)
        input_path = r"./file/input/221.xlsx"

        # 📤 output folder
        output_dir = r"./file/output"

        import os
        os.makedirs(output_dir, exist_ok=True)

        # อ่านไฟล์
        input_df_from_file = pd.read_excel(input_path)

        # รัน algorithm
        grading_results = hybrid_grading_algorithm(
            input_df_from_file,
            grade_symbols=['A', 'B+', 'B', 'C+', 'C', 'D+', 'D']
        )

        print("\n=== RESULT ===")
        print(grading_results)

Running in local environment...
Running K-means grading...
KMeans Grade Boundaries (derived from assigned scores): [(np.float64(86.5), np.float64(99.1), 'A'), (np.float64(77.5), np.float64(85.36), 'B+'), (np.float64(69.78999999999999), np.float64(76.64), 'B'), (np.float64(61.96), np.float64(68.5), 'C+'), (np.float64(55.5), np.float64(61.5), 'C'), (np.float64(44.33), np.float64(53.5), 'D+'), (np.float64(35.0), np.float64(43.5), 'D')]
K-means Omega: 0.143
K-means grading output saved to './file/output/kmean/graded_scores_kmeans.xlsx'
Running Z-score grading...
Zscore Grade Boundaries (derived from assigned scores): [(np.float64(90.5), np.float64(99.1), 'A'), (np.float64(80.89), np.float64(89.5), 'B+'), (np.float64(71.63), np.float64(80.5), 'B'), (np.float64(62.5), np.float64(71.57), 'C+'), (np.float64(53.5), np.float64(62.14), 'C'), (np.float64(44.33), np.float64(53.0), 'D+'), (np.float64(35.0), np.float64(43.5), 'D')]
Z-score Omega: 0.178
Z-score grading output saved to './file/output/z